In [ ]:
# libraries
import pandas as pd
import re

In [17]:
df = pd.read_csv("../data/raw/Mobile Reviews Sentiment null.csv")

In [18]:
df.head()

,review_id,customer_name,age,brand,model,price_usd,price_local,currency,exchange_rate_to_usd,rating,...,language,review_date,verified_purchase,battery_life_rating,camera_rating,performance_rating,design_rating,display_rating,helpful_votes,source
0,1,Aryan Maharaj,45,Realme,Realme 12 Pro,337.31,₹27996.73,INR,83.00,2.0,...,Hindi,11/6/2023,True,1,1,3,2,1,1,Amazon
1,2,Davi Miguel Sousa,18,Realme,Realme 12 Pro,307.78,R$1754.35,BRL,5.70,4.0,...,Portuguese,3/30/2023,True,3,2,4,3,2,5,Flipkart
2,3,Pahal Balay,27,Google,Pixel 6,864.53,₹71755.99,INR,83.00,4.0,...,Hindi,12/7/2022,True,3,5,3,2,4,8,AliExpress
3,4,David Guzman,19,Xiaomi,Redmi Note 13,660.94,د.إ2425.65,AED,3.67,3.0,...,English,3/11/2025,False,1,3,2,1,2,3,Amazon
4,5,Yago Leão,38,Motorola,Edge 50,792.13,R$4515.14,BRL,5.70,3.0,...,Portuguese,9/29/2023,True,3,3,2,2,1,0,BestBuy


In [19]:
str_cols = ['brand', 'model', 'currency', 'sentiment', 'country', 'language', 'source']

for col in str_cols:
    irregular = df[col].dropna().unique()
    irregular = [v for v in irregular if re.search(r'[^a-zA-Z\s,]', v)]
    print(col, '→', irregular)

brand → []
model → ['Realme 12 Pro', 'Pixel 6', 'Redmi Note 13', 'Edge 50', 'iPhone 14', 'OnePlus 12', 'OnePlus Nord 3', 'Mi 13 Pro', 'Realme Narzo 70', 'Galaxy Note 20', 'Pixel 7a', 'Razr 40', 'Poco X6', 'iPhone 15 Pro', 'Pixel 8', 'iPhone 13', 'Galaxy S24', 'Galaxy A55', 'OnePlus 11R']
currency → []
sentiment → []
country → []
language → []
source → []


In [20]:
df.drop(columns=['review_id', 'customer_name'], inplace=True)

In [21]:
for col in str_cols:
    df[col] = (
        df[col]
        .str.strip()
        .str.replace(r'\s+', ' ', regex=True)
        .str.title()
    )

In [22]:
df['currency'] = df['currency'].str.upper()

In [23]:
df['price_local'] = (
    df['price_local'].astype('object')
      .str.replace(',', '', regex=False)              # remove thousand separators
      .str.extract(r'(\d+(?:\.\d+)?)\s*$')[0]         # number at the end only
      .astype(float)
)

In [24]:
df['review_date'] = pd.to_datetime(df['review_date'], format='%m/%d/%Y')

In [25]:
df['age'] = df['age'].astype('int16')

In [27]:
rating_cols = ['battery_life_rating', 'camera_rating', 'performance_rating',
               'design_rating', 'display_rating', 'helpful_votes']
df[rating_cols] = df[rating_cols].astype('int8' )

In [28]:
print(df.dtypes)
print(df['price_local'].describe())

age                              int16
brand                              str
model                              str
price_usd                      float64
price_local                    float64
currency                           str
exchange_rate_to_usd           float64
rating                         float64
sentiment                          str
country                            str
language                           str
review_date             datetime64[us]
verified_purchase                 bool
battery_life_rating               int8
camera_rating                     int8
performance_rating                int8
design_rating                     int8
display_rating                    int8
helpful_votes                     int8
source                             str
dtype: object
count     47569.000000
mean       8289.618042
std       20385.555363
min         141.090000
25%         594.120000
50%        1059.780000
75%        2916.290000
max      124397.080000
Name: price_local, dty

In [29]:
str_cols = ['brand', 'model', 'country', 'language', 'currency', 'source', 'sentiment']
num_cols = ['age', 'price_usd', 'price_local', 'exchange_rate_to_usd', 'rating',
            'battery_life_rating', 'camera_rating', 'performance_rating',
            'design_rating', 'display_rating', 'helpful_votes']

new_order = ['review_date', 'verified_purchase'] + str_cols + num_cols
df = df[new_order]
print(df.shape)

(50000, 20)


In [30]:
# fill price_usd from local price / exchange rate
mask = df['price_usd'].isna() & df['price_local'].notna()
df.loc[mask, 'price_usd'] = df.loc[mask, 'price_local'] / df.loc[mask, 'exchange_rate_to_usd']

print("filled:", mask.sum())
print("price_usd nulls left:", df['price_usd'].isna().sum())

filled: 2327
price_usd nulls left: 123


In [31]:
# model-level median for the remaining rows
df['price_usd'] = df['price_usd'].fillna(df.groupby('model')['price_usd'].transform('median'))

# Step 3: fill price_local from usd × rate (keeps both columns consistent)
mask = df['price_local'].isna()
df.loc[mask, 'price_local'] = (df.loc[mask, 'price_usd'] * df.loc[mask, 'exchange_rate_to_usd']).round(2)

print(df[['price_usd', 'price_local']].isna().sum())

price_usd      0
price_local    0
dtype: int64


In [ ]:
# other rating corrections based on the average of the sub-ratings
sub = ['battery_life_rating', 'camera_rating', 'performance_rating',
       'design_rating', 'display_rating']

mask = df['rating'].isna()
df.loc[mask, 'rating'] = df.loc[mask, sub].mean(axis=1).round().clip(1, 5)

print("filled:", mask.sum())
print("rating nulls left:", df['rating'].isna().sum())
print(df['rating'].value_counts().sort_index())

filled: 2453
rating nulls left: 0
rating
1.0     6539
2.0     9944
3.0    12511
4.0    13882
5.0     7124
Name: count, dtype: int64


In [34]:
# most common sentiment per rating (mode ignores NaN)
mode_map = df.groupby('rating')['sentiment'].agg(lambda s: s.mode().iloc[0])
print(mode_map)

mask = df['sentiment'].isna()
df.loc[mask, 'sentiment'] = df.loc[mask, 'rating'].map(mode_map)

print("filled:", mask.sum())
print("sentiment nulls left:", df['sentiment'].isna().sum())
print(df['sentiment'].value_counts())

rating
1.0    Negative
2.0     Neutral
3.0    Positive
4.0    Positive
5.0    Positive
Name: sentiment, dtype: str
filled: 2445
sentiment nulls left: 0
sentiment
Positive    27851
Neutral     12437
Negative     9712
Name: count, dtype: int64


In [36]:
df['source'] = df['source'].fillna('Unknown')

print(df['source'].value_counts())
print("total nulls in dataset:", df.isna().sum().sum())

source
Ebay          9573
Aliexpress    9531
Amazon        9513
Flipkart      9481
Bestbuy       9454
Unknown       2448
Name: count, dtype: int64
total nulls in dataset: 0


In [37]:
df

,review_date,verified_purchase,brand,model,country,language,currency,source,sentiment,age,price_usd,price_local,exchange_rate_to_usd,rating,battery_life_rating,camera_rating,performance_rating,design_rating,display_rating,helpful_votes
0,2023-11-06,True,Realme,Realme 12 Pro,India,Hindi,INR,Amazon,Negative,45,337.31,27996.73,83.00,2.0,1,1,3,2,1,1
1,2023-03-30,True,Realme,Realme 12 Pro,Brazil,Portuguese,BRL,Flipkart,Positive,18,307.78,1754.35,5.70,4.0,3,2,4,3,2,5
2,2022-12-07,True,Google,Pixel 6,India,Hindi,INR,Aliexpress,Positive,27,864.53,71755.99,83.00,4.0,3,5,3,2,4,8
3,2025-03-11,False,Xiaomi,Redmi Note 13,Uae,English,AED,Amazon,Positive,19,660.94,2425.65,3.67,3.0,1,3,2,1,2,3
4,2023-09-29,True,Motorola,Edge 50,Brazil,Portuguese,BRL,Bestbuy,Neutral,38,792.13,4515.14,5.70,3.0,3,3,2,2,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49995,2023-08-17,False,Realme,Realme 12 Pro,Australia,English,AUD,Bestbuy,Positive,18,526.10,804.93,1.53,4.0,3,3,4,3,5,4
49996,2023-09-23,False,Samsung,Galaxy Note 20,Australia,English,AUD,Aliexpress,Neutral,28,911.85,1395.13,1.53,3.0,3,2,4,1,3,2
49997,2023-04-04,True,Apple,Iphone Se,Uae,English,AED,Amazon,Positive,21,1155.32,4240.02,3.67,4.0,2,3,4,4,5,4
49998,2025-07-11,True,Realme,Realme Narzo 70,Usa,English,USD,Unknown,Positive,31,459.04,459.04,1.00,3.0,1,2,2,1,3,6


In [38]:
print("duplicates:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)

duplicates: 0


In [39]:
# outliers: check numeric columns
print(df[num_cols].describe().T[['min', '25%', '50%', '75%', 'max']])

                         min      25%       50%       75%        max
age                    18.00   23.000    29.000    36.000      65.00
price_usd             180.02  451.040   637.125   901.110    1499.89
price_local           141.09  593.985  1060.260  2921.895  124397.08
exchange_rate_to_usd    0.78    1.000     1.530     5.700      83.00
rating                  1.00    2.000     3.000     4.000       5.00
battery_life_rating     1.00    1.000     3.000     4.000       5.00
camera_rating           1.00    1.000     3.000     4.000       5.00
performance_rating      1.00    1.000     3.000     4.000       5.00
design_rating           1.00    1.000     3.000     4.000       5.00
display_rating          1.00    1.000     3.000     4.000       5.00
helpful_votes           0.00    2.000     3.000     5.000      17.00


In [40]:
# IQR outliers on price_usd, per count only (don't drop yet)
q1, q3 = df['price_usd'].quantile([.25, .75])
iqr = q3 - q1
out = df[(df['price_usd'] < q1 - 1.5*iqr) | (df['price_usd'] > q3 + 1.5*iqr)]
print("price_usd outliers:", len(out))

price_usd outliers: 0


In [41]:
# sanity checks
print("age range:", df['age'].min(), df['age'].max())
print("ratings outside 1-5:", ((df['rating'] < 1) | (df['rating'] > 5)).sum())

age range: 18 65
ratings outside 1-5: 0


In [44]:
df.to_csv('../data/cleaned/mobile_reviews_cleaned.csv', index=False)

In [45]:
chk = pd.read_csv('../data/cleaned/mobile_reviews_cleaned.csv', parse_dates=['review_date'])
print(chk.shape, chk.isna().sum().sum())

(50000, 20) 0
